# Name That LLM — frozen v3 CUDA transfer (Colab launcher)

**Purpose:** Run the existing, *frozen* paired **v1 control vs DCT-v3** transfer gate on a suitable Colab CUDA GPU. This does **not** start from the BF16 CPU shadow checkpoint; it runs the frozen CUDA lineage from scratch.

**Before running:** In Colab select **Runtime → Change runtime type → GPU**. The free tier may give a T4 with less than 15 GiB usable VRAM; this notebook **fails closed** on undersized hardware. Even 15 GiB is only the repository's *minimum preflight*, not a promise that training fits. A GPU with 24+ GiB is preferable.

**Scientific guards:** pinned source commit, source-pool SHA, Qwen revision, 24 layers, fixed DCT modes [0,1], frozen heldout sources and gates. No official competition rows, no Public LB, no Kaggle submission. Never change the frozen configuration just to make Colab fit.

**Required secret:** In Colab Secrets (key icon), add `KAGGLE_API_TOKEN` and enable notebook access. Do **not** paste the token into a cell or commit it.

**Old-run gate:** Existing self-hosted scientific run `41580819667` must be cancelled before executing another exact paired transfer. This notebook verifies it. A queued or running old run blocks execution.

**Runtime limits:** Free Colab GPU is not guaranteed, and free runtimes can end before a long two-model training experiment finishes. The current paired evaluator saves the v3 checkpoint on *completion*, not at every microbatch. There is no automatic resume of that exact CUDA train here.

In [ ]:
import torch, subprocess, json
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. Runtime → Change runtime type → GPU.")
p = torch.cuda.get_device_properties(0)
vram_gib = p.total_memory / 1024**3
print(json.dumps({"gpu": p.name, "vram_gib": round(vram_gib, 2), "torch": torch.__version__, "cuda": torch.version.cuda}, indent=2))
if vram_gib < 15.0:
    raise RuntimeError(
        f"Frozen v3 GPU preflight requires >=15 GiB VRAM; allocated {vram_gib:.2f} GiB. "
        "Do not substitute QLoRA/layer reduction and call it v3. "
        "Try a larger Colab GPU or treat a T4 experiment as a separately labeled proxy."
    )
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import json, urllib.request
OLD_RUN = 41580819667
url = f"https://api.github.com/repos/kolihhan/name-kaggle/actions/runs/{OLD_RUN}"
req = urllib.request.Request(url, headers={"User-Agent": "ntl-v3-colab-preflight", "Accept": "application/vnd.github+json"})
with urllib.request.urlopen(req, timeout=30) as response:
    run = json.load(response)
print("Existing frozen CUDA experiment:", run["html_url"], "status:", run["status"], "conclusion:", run["conclusion"])
if run["status"] != "completed" or run["conclusion"] not in ("cancelled",):
    raise RuntimeError(
        "Cancel old run 41580819667 in GitHub Actions before running Colab. "
        "If it completed/scored already, do not repeat the frozen scientific gate."
    )

In [ ]:
import subprocess
from pathlib import Path
PINNED_SOURCE_COMMIT = "b86e8cb68631a18a2a4d1342658ecdca740a3f2b"
REPO = Path("/content/name-kaggle")
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "init", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "remote", "add", "origin", "https://github.com/kolihhan/name-kaggle.git"], check=True)
    subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "init", "--cone"], check=True)
    subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "set", "scripts", "tests", "artifacts/candidates/dct-temporal-fingerprint-v3"], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth=1", "--filter=blob:none", "origin", PINNED_SOURCE_COMMIT], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == PINNED_SOURCE_COMMIT, (actual, PINNED_SOURCE_COMMIT)
for forbidden in ("data/train.csv", "data/test.csv", "data/sample_submission.parquet"):
    assert not (REPO / forbidden).exists(), f"Official competition file unexpectedly materialized: {forbidden}"
print("Pinned v3 scientific source:", actual, "| sparse checkout; no official competition files.")

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers==4.57.1", "peft==0.17.1", "tokenizers==0.22.1",
     "huggingface-hub==0.36.0", "kaggle==2.2.5",
     "safetensors", "pandas", "numpy", "scipy", "scikit-learn",
     "pyarrow", "pytest", "six"],
    check=True,
)
import transformers, peft
assert transformers.__version__ == "4.57.1"
assert peft.__version__ == "0.17.1"
print("Runtime dependencies pinned. Torch preserved:", __import__("torch").__version__)

In [ ]:
import os, subprocess, sys
env = dict(os.environ, PYTHONPATH=str(REPO))
suite = [
  "tests/test_dct_v3_transfer_split.py",
  "tests/test_dct_temporal_pool.py",
  "tests/test_materialize_dct_v3_worker.py",
  "tests/test_dct_v3_source_eval.py",
  "tests/test_dct_v3_local_transfer.py",
]
subprocess.run([sys.executable, "-m", "pytest", "-q", *suite], cwd=REPO, env=env, check=True)
print("Static v3 contract verified on Colab; no model evaluation performed.")

In [ ]:
import os
from google.colab import userdata
token = userdata.get("KAGGLE_API_TOKEN")
if not token:
    raise RuntimeError("Add KAGGLE_API_TOKEN to Colab Secrets and enable notebook access.")
TRAIN_ENV = dict(os.environ, KAGGLE_API_TOKEN=token, PYTHONPATH=str(REPO), TOKENIZERS_PARALLELISM="false")
del token
print("Kaggle credentials loaded into process environment (not printed).")

## Exact frozen paired transfer — execute once

The cell below executes `scripts/dct_v3_local_transfer.py` exactly as the repo's self-hosted GPU workflow does. It downloads the **SHA-locked external public pool** using the Kaggle token, verifies the pinned Qwen model revision, trains v1 and v3 on the unchanged source split, and evaluates the frozen source-transfer gate.

**This is not a lightweight smoke test.** Only run it when your allocated GPU has sufficient memory and runtime. The full v1+v3 training may outlast a free Colab session. No Kaggle submission is made.

In [ ]:
import subprocess, sys
WORK = Path("/content/v3-frozen-paired-transfer")
CHECKPOINT = WORK / "v3_stage1_checkpoint"
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [sys.executable, "scripts/dct_v3_local_transfer.py",
     "--work-dir", str(WORK), "--checkpoint-dir", str(CHECKPOINT)],
    cwd=REPO, env=TRAIN_ENV, check=True
)
result = json.loads((WORK / "paired_transfer_result.json").read_text())
assert result["official_rows_scored"] == 0
assert result["competition_test_rows_used"] == 0
assert result["public_lb_tuning"] is False
print("V3 transfer gate:", result["decision"], "| submission not authorized by this notebook.")

In [ ]:
from google.colab import drive
import shutil
from pathlib import Path
drive.mount("/content/drive")
DEST = Path("/content/drive/MyDrive/name-kaggle-v3-colab") / PINNED_SOURCE_COMMIT[:12]
if not (WORK / "paired_transfer_result.json").exists():
    raise RuntimeError("No completed frozen transfer result to preserve. Do not label an incomplete run PASS/FAIL.")
DEST.parent.mkdir(parents=True, exist_ok=True)
if DEST.exists():
    raise FileExistsError(f"Refusing to overwrite an existing frozen result: {DEST}")
shutil.copytree(WORK, DEST)
print("Preserved exact completed result and trainable-only checkpoint at:", DEST)
print("Manual review required before any Stage 2 / Kaggle submission.")